Used dtataset: **Ames Housing Dataset**

In [4]:
import kagglehub
import pandas as pd
import numpy as np
import os

if(not os.path.isdir('data')):
    path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset", output_dir='./data')

    print("Path to dataset:", path)

df = pd.read_csv('./data/AmesHousing.csv')

print(df.shape)  # (2930, 82)
df.info()

(2930, 82)
<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style   

In [19]:
df.groupby('Functional')['SalePrice'].mean().sort_values()

Functional
Sal      31550.000000
Maj2     91122.666667
Sev      95750.000000
Min2    147701.885714
Min1    151094.615385
Mod     151640.085714
Maj1    152751.263158
Typ     183389.953446
Name: SalePrice, dtype: float64

# Przygotowanie danych

* zamienienie wartosci NaN na 0 itp
* Zamiana Stringów
* Standaryzacja (StandardScaler)
* Usuwanie outliersów (Z-score, IQR method, visually)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

def clean_data(df:pd.DataFrame, train_settings = None):

    df.drop(columns=['Order', 'PID', 'Garage Yr Blt'], inplace=True)

    # Fill categorical "absent" columns with 'None'
    none_cols = [ 'Alley', 'Mas Vnr Type', 'Bsmt Qual', 'Bsmt Cond',
                  'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2',
                  'Fireplace Qu', 'Garage Type', 'Garage Finish',
                  'Garage Qual', 'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature' ]
    df[none_cols] = df[none_cols].fillna('None')

    # Fill numeric absent columns with 0
    zero_cols = [ 'Lot Frontage', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2',
                  'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath',
                  'Garage Cars', 'Garage Area' ]
    df[zero_cols] = df[zero_cols].fillna(0)

    df['Electrical'] = df['Electrical'].fillna('SBrkr')

    # Ordinal string columns that will be changed by LabelEncoder
    ordinal_cols = {
        'Lot Shape':        ['IR3', 'IR2', 'IR1', 'REG'],
        'Utilities':        ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
        'Land Slope':       ['Gtl', 'Mod', 'Sev'],
        'Exter Qual':       ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
        'Exter Cond':       ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
        'Bsmt Qual':        ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
        'Bsmt Cond':        ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
        'Bsmt Exposure':    ['None', 'No', 'Mn', 'Av', 'Gd'],
        'BsmtFin Type 1':   ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
        'BsmtFin Type 2':   ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
        'Central Air':      ['N', 'Y'],
        'Electrical':       ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr'],
        'KitchenQual':      ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
        'Functional':       ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    }

    # Non-ordinal string columns that will be One-Hot-encoded or Target-encoded
    non_ordinal_cols = [ 'MS Zoning', 'Street', 'Alley', 'Land Contour',
                         'Lot Config', 'Neighborhood', 'Condition 1', 'Condition 2'
                         'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
                         'Exterior 1', 'Exterior 2', 'Mas Vnr Type', 'Foundation',
                         'Heating'
                          ]

    # Number columns that should not be interpreted as continuous
    int_cols_to_categories = [ 'MS SubClass' ]

    # Applying ordinal encoding:
    for col, order in ordinal_cols.items():
        df[col].fillna('None').astype(str)
        df[col] = df[col].astype(str)
        enc = OrdinalEncoder(
        categories=[order],
        handle_unknown='use_encoded_value',
        unknown_value=-1         
        )
        df[col] = enc.fit_transform(df[[col]]).ravel()
    
    return df #median


# Skonczyłem na Functional

In [6]:
df = clean_data(df)
df.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,20,RL,141.0,31770,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,None,Reg,Lvl,AllPub,Corner,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,None,IR1,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 79 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MS SubClass      2930 non-null   int64  
 1   MS Zoning        2930 non-null   str    
 2   Lot Frontage     2930 non-null   float64
 3   Lot Area         2930 non-null   int64  
 4   Street           2930 non-null   str    
 5   Alley            2930 non-null   str    
 6   Lot Shape        2930 non-null   str    
 7   Land Contour     2930 non-null   str    
 8   Utilities        2930 non-null   str    
 9   Lot Config       2930 non-null   str    
 10  Land Slope       2930 non-null   str    
 11  Neighborhood     2930 non-null   str    
 12  Condition 1      2930 non-null   str    
 13  Condition 2      2930 non-null   str    
 14  Bldg Type        2930 non-null   str    
 15  House Style      2930 non-null   str    
 16  Overall Qual     2930 non-null   int64  
 17  Overall Cond     2930 non

# Wizualizacja danych

* Wykresy pokazujące zależność między cechami a ceną. 
* TABELE
* INNE WYKRESY